# 03 — Feature Engineering

**Projeto:** Financial Behavior Intelligence  
**Objetivo:** Construir um DataFrame agregado por usuário com features que descrevem o comportamento financeiro. Este arquivo será o input do modelo de ML.

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/transactions_clean.csv', parse_dates=['Date'])
print('Shape:', df.shape)
df.head(3)

## 1. Agregações Mensais por Usuário

In [ ]:
df['Period'] = df['Date'].dt.to_period('M')

monthly = df.pivot_table(
    index=['User_ID', 'Period'],
    columns='Transaction Type',
    values='Amount',
    aggfunc='sum',
    fill_value=0
).reset_index()

monthly.columns.name = None
monthly['saldo_mensal'] = monthly.get('credit', 0) - monthly.get('debit', 0)
monthly.head()

## 2. Features por Usuário

In [ ]:
# --- Bloco 2a: Features de saldo e receita/despesa ---
feat_saldo = monthly.groupby('User_ID').agg(
    avg_monthly_debit   = ('debit',        'mean'),
    avg_monthly_credit  = ('credit',       'mean'),
    avg_saldo           = ('saldo_mensal', 'mean'),
    std_saldo           = ('saldo_mensal', 'std'),
    spending_volatility = ('debit',        'std'),
    pct_meses_negativo  = ('saldo_mensal', lambda x: round((x < 0).mean(), 3)),
    total_meses         = ('saldo_mensal', 'count')
).reset_index()

# Taxa de poupança média = (receita - despesa) / receita
feat_saldo['savings_rate'] = (
    (feat_saldo['avg_monthly_credit'] - feat_saldo['avg_monthly_debit']) /
    feat_saldo['avg_monthly_credit'].replace(0, np.nan)
).round(3)

feat_saldo.head()

In [ ]:
# --- Bloco 2b: Features de categoria (apenas débitos) ---
debits = df[df['Transaction Type'] == 'debit']

total_by_user = debits.groupby('User_ID')['Amount'].sum().rename('total_debit')

# % do gasto na principal categoria
top_cat = (debits.groupby(['User_ID', 'Category'])['Amount'].sum()
           .reset_index()
           .sort_values('Amount', ascending=False)
           .groupby('User_ID')
           .first()
           .reset_index()
           .rename(columns={'Category': 'top_category', 'Amount': 'top_category_spend'}))

top_cat = top_cat.merge(total_by_user, on='User_ID')
top_cat['top_category_pct'] = (top_cat['top_category_spend'] / top_cat['total_debit']).round(3)

# Categorias fixas (recorrentes)
FIXED_CATS = ['Rent', 'Phone Bill', 'Internet Bill', 'Insurance', 'Utilities']
fixed = (debits[debits['Category'].isin(FIXED_CATS)]
         .groupby('User_ID')['Category']
         .nunique()
         .rename('num_fixed_expenses')
         .reset_index())

# Tem investimento?
has_inv = (debits[debits['Category'] == 'Investment']
           .groupby('User_ID')['Amount']
           .sum()
           .rename('total_investment')
           .reset_index())
has_inv['has_investment'] = 1

print('Features de categoria OK')

In [ ]:
# --- Bloco 2c: Uso de cartão de crédito ---
credit_card_use = (df.groupby(['User_ID', 'Account Name'])
                   .size()
                   .unstack(fill_value=0)
                   .reset_index())

credit_card_use['total_transactions'] = credit_card_use.iloc[:, 1:].sum(axis=1)
if 'credit card' in credit_card_use.columns:
    credit_card_use['credit_card_ratio'] = (
        credit_card_use['credit card'] / credit_card_use['total_transactions']
    ).round(3)
else:
    credit_card_use['credit_card_ratio'] = 0

feat_account = credit_card_use[['User_ID', 'credit_card_ratio']]
feat_account.head()

## 3. Consolidar todas as features

In [ ]:
features = feat_saldo.copy()

for other in [top_cat[['User_ID', 'top_category', 'top_category_pct']],
              fixed,
              has_inv[['User_ID', 'has_investment', 'total_investment']],
              feat_account]:
    features = features.merge(other, on='User_ID', how='left')

features['has_investment']    = features['has_investment'].fillna(0).astype(int)
features['total_investment']  = features['total_investment'].fillna(0)
features['num_fixed_expenses'] = features['num_fixed_expenses'].fillna(0)

print('Shape final de features:', features.shape)
features.head()

## 4. Salvar

In [ ]:
features.to_csv('../data/processed/user_features.csv', index=False)
print('Features salvas em data/processed/user_features.csv')
features.dtypes